# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you in loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library. We'll follow a step-by-step process based on the Croissant schema and exclusively reference data entities by their `@id` attributes for clarity and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install and import mlcroissant if not already installed
!pip install mlcroissant --quiet

## 1. Data Loading
Let's load the dataset metadata and initialize the `mlcroissant` Dataset object.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print out basic metadata from the Dataset object
print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")

## 2. Data Overview
Let's inspect what record sets are available and their field composition by `@id`. The `mlcroissant.Dataset` API allows us to retrieve record set definitions as described in the Croissant metadata. We'll list all record sets, their `@id`s, and their field `@id`s.

In [ ]:
# Enumerate available record sets and their fields by @id
print("Available record sets and fields:")

record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets found in the dataset. Please check the dataset metadata.")
else:
    for rs in record_sets:
        print(f"- RecordSet @id: {rs.id}")
        print(f"  Name: {rs.name if hasattr(rs, 'name') else ''}")
        print(f"  Description: {rs.description if hasattr(rs, 'description') else ''}")
        print("  Fields: [")
        for f in rs.fields:
            print(f"    - Field @id: {f.id} (name: {f.name if hasattr(f,'name') else ''})")
        print("  ]\n")

## 3. Data Extraction
Let's extract records from each record set. We will use their `@id` as reference and load the data into DataFrames. We'll print available column names (fields) for a selected record set.

In [ ]:
# Get record set @id list
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

# Create a dictionary of DataFrames for each record set
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    # Handle case of empty records
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        dataframes[record_set_id] = pd.DataFrame()

# For demonstration, print columns for the first available non-empty record set
main_record_set_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rs_id
        break

if main_record_set_id:
    print(f"Primary record set id for analysis: {main_record_set_id}\n")
    print(f"Columns (field @id, as named in data):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No non-empty record set data found.")

## 4. Exploratory Data Analysis (EDA)
Let's select a representative numeric field by `@id` (from the previous overview) and run standard EDA operations: filtering, normalizing, and grouping.

**Replace `<numeric_field_id>` and `<group_field_id>` below with actual field `@id` strings from your data.**

In [ ]:
# Example: Suppose one field is 'cr:MW' (molecular weight), and group field is 'cr:SampleType' (use actual @id)
numeric_field_id = '<numeric_field_id>'      # Example: 'cr:MW'
group_field_id = '<group_field_id>'          # Example: 'cr:SampleType'

# If you know the field names from the print above, update these lines for real analysis.
df = dataframes.get(main_record_set_id, pd.DataFrame())
if numeric_field_id not in df.columns or df.empty:
    print(f"Field '{numeric_field_id}' not found in record set '{main_record_set_id}'. Please set numeric_field_id to a valid column name.")
else:
    # Drop NA and ensure float type
    df_eda = df.dropna(subset=[numeric_field_id]).copy()
    df_eda[numeric_field_id] = pd.to_numeric(df_eda[numeric_field_id], errors='coerce')
    threshold = df_eda[numeric_field_id].mean()  # use mean as an example
    filtered_df = df_eda[df_eda[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalization (z-score)
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} (z-score) for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Grouping
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
    else:
        print(f"Grouping field '{group_field_id}' not found in DataFrame columns. Skipping group-by analysis.")

## 5. Visualization
Let's plot distributions or relationships in the data using the chosen numeric and group fields.

**Make sure to set `numeric_field_id` and `group_field_id` to existing column names.**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Check DataFrame and fields before plotting
if df.empty or numeric_field_id not in df.columns:
    print(f"Cannot plot: DataFrame empty or field '{numeric_field_id}' not present.")
else:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].astype(float), kde=True)
    plt.title(f"Distribution of {numeric_field_id} in record set {main_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If a grouping field is available, plot boxplot/grouped bar
    if group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id].astype(float))
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, you have learned how to use `mlcroissant` with a FAIR^2 dataset by referencing all entity `@id` values, inspected available data structures, extracted tabular data, applied exploratory data analysis, and generated initial visualizations. For in-depth analysis, use actual field `@id` values from the metadata overview, and refer to the Croissant specification to enrich your workflow.

To extend this notebook further, you could:
- Join across record sets by foreign key field `@id` if applicable
- Engineer new features using the field `@id`s
- Save processed DataFrames for downstream ML tasks

For more information about the dataset and its fields, refer to the [original Croissant metadata file](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).